# Day 62 — Responsible AI & Governance
### Bias detection, fairness metrics, EU AI Act, hallucination mitigation, model card

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

np.random.seed(42)
print("Setup ready")

Setup ready


## 2. Types of Bias in AI Systems

In [2]:
bias_types = pd.DataFrame([
    {
        "bias_type": "Historical Bias",
        "description": "Training data reflects past societal inequalities",
        "example": "Hiring model trained on historical data that favoured men for tech roles",
        "mitigation": "Rebalance dataset, apply fairness constraints during training"
    },
    {
        "bias_type": "Representation Bias",
        "description": "Certain groups underrepresented in training data",
        "example": "Facial recognition trained mostly on light-skinned faces performs poorly on darker skin",
        "mitigation": "Collect more diverse data, oversample underrepresented groups"
    },
    {
        "bias_type": "Measurement Bias",
        "description": "The way data is collected or labelled introduces bias",
        "example": "Medical AI trained on data from wealthy hospitals, poor generalisation to rural clinics",
        "mitigation": "Audit data collection pipelines, use multiple diverse data sources"
    },
    {
        "bias_type": "Aggregation Bias",
        "description": "Model built for the general population fails specific subgroups",
        "example": "Diabetes risk model trained on aggregated data ignores different risk patterns by ethnicity",
        "mitigation": "Build subgroup-specific models or use stratified evaluation"
    },
    {
        "bias_type": "Evaluation Bias",
        "description": "Model evaluated on benchmarks that don't reflect real-world use",
        "example": "NLP model scores well on English benchmarks but fails on non-English speakers",
        "mitigation": "Evaluate on diverse, real-world representative test sets"
    },
    {
        "bias_type": "Deployment Bias",
        "description": "Model used in a context different from what it was built for",
        "example": "Sentiment model trained on product reviews deployed for medical text analysis",
        "mitigation": "Domain-specific validation before deployment, monitor in production"
    }
])

bias_types

,bias_type,description,example,mitigation
0,Historical Bias,Training data reflects past societal inequalities,Hiring model trained on historical data that f...,"Rebalance dataset, apply fairness constraints ..."
1,Representation Bias,Certain groups underrepresented in training data,Facial recognition trained mostly on light-ski...,"Collect more diverse data, oversample underrep..."
2,Measurement Bias,The way data is collected or labelled introduc...,Medical AI trained on data from wealthy hospit...,"Audit data collection pipelines, use multiple ..."
3,Aggregation Bias,Model built for the general population fails s...,Diabetes risk model trained on aggregated data...,Build subgroup-specific models or use stratifi...
4,Evaluation Bias,Model evaluated on benchmarks that don't refle...,NLP model scores well on English benchmarks bu...,"Evaluate on diverse, real-world representative..."
5,Deployment Bias,Model used in a context different from what it...,Sentiment model trained on product reviews dep...,"Domain-specific validation before deployment, ..."


## 3. Fairness Metrics — Hands-On Calculation

In [3]:
# simulate a loan approval model's predictions on two demographic groups
np.random.seed(42)
n = 200

# group A (majority) — higher approval rate in this biased model
group_a = pd.DataFrame({
    "group": "A",
    "actual": np.random.choice([0, 1], n, p=[0.4, 0.6]),
    "predicted": np.random.choice([0, 1], n, p=[0.3, 0.7])
})

# group B (minority) — lower approval rate due to bias
group_b = pd.DataFrame({
    "group": "B",
    "actual": np.random.choice([0, 1], n, p=[0.4, 0.6]),
    "predicted": np.random.choice([0, 1], n, p=[0.55, 0.45])
})

df = pd.concat([group_a, group_b], ignore_index=True)

def fairness_metrics(group_df, group_name):
    tp = ((group_df.predicted == 1) & (group_df.actual == 1)).sum()
    fp = ((group_df.predicted == 1) & (group_df.actual == 0)).sum()
    tn = ((group_df.predicted == 0) & (group_df.actual == 0)).sum()
    fn = ((group_df.predicted == 0) & (group_df.actual == 1)).sum()

    approval_rate = group_df.predicted.mean()
    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0  # true positive rate = recall
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0  # false positive rate

    return {
        "group": group_name,
        "approval_rate": round(approval_rate, 3),
        "true_positive_rate": round(tpr, 3),
        "false_positive_rate": round(fpr, 3),
        "total": len(group_df)
    }

metrics_a = fairness_metrics(group_a, "Group A (majority)")
metrics_b = fairness_metrics(group_b, "Group B (minority)")

results = pd.DataFrame([metrics_a, metrics_b])
print(results.to_string(index=False))

print(f"\nDemographic Parity Gap:  {abs(metrics_a['approval_rate'] - metrics_b['approval_rate']):.3f}")
print(f"Equal Opportunity Gap:   {abs(metrics_a['true_positive_rate'] - metrics_b['true_positive_rate']):.3f}")
print(f"\nFairness threshold (common standard): gap < 0.1")
print(f"Demographic Parity:  {'PASS' if abs(metrics_a['approval_rate'] - metrics_b['approval_rate']) < 0.1 else 'FAIL'}")
print(f"Equal Opportunity:   {'PASS' if abs(metrics_a['true_positive_rate'] - metrics_b['true_positive_rate']) < 0.1 else 'FAIL'}")

             group  approval_rate  true_positive_rate  false_positive_rate  total
Group A (majority)          0.705               0.700                0.711    200
Group B (minority)          0.410               0.427                0.382    200

Demographic Parity Gap:  0.295
Equal Opportunity Gap:   0.273

Fairness threshold (common standard): gap < 0.1
Demographic Parity:  FAIL
Equal Opportunity:   FAIL


## 4. EU AI Act — What Practitioners Need to Know

In [4]:
eu_ai_act = pd.DataFrame([
    {
        "risk_tier": "Unacceptable Risk",
        "description": "Banned outright",
        "examples": "Social scoring by governments, real-time biometric surveillance in public spaces, AI that exploits vulnerabilities of specific groups",
        "requirements": "PROHIBITED — cannot be built or deployed in the EU"
    },
    {
        "risk_tier": "High Risk",
        "description": "Heavily regulated, must comply before deployment",
        "examples": "CV screening tools, credit scoring, medical diagnosis, critical infrastructure, law enforcement AI",
        "requirements": "Conformity assessment, risk management system, data governance, human oversight, transparency, accuracy/robustness standards"
    },
    {
        "risk_tier": "Limited Risk",
        "description": "Transparency obligations only",
        "examples": "Chatbots, deepfakes, AI-generated content",
        "requirements": "Must disclose AI interaction to users. Deepfakes must be labelled."
    },
    {
        "risk_tier": "Minimal Risk",
        "description": "No specific requirements",
        "examples": "Spam filters, AI in video games, recommendation systems",
        "requirements": "Encouraged to follow voluntary codes of conduct"
    },
])

pd.set_option('display.max_colwidth', 60)
eu_ai_act[["risk_tier", "examples", "requirements"]]

,risk_tier,examples,requirements
0,Unacceptable Risk,"Social scoring by governments, real-time biometric surve...",PROHIBITED — cannot be built or deployed in the EU
1,High Risk,"CV screening tools, credit scoring, medical diagnosis, c...","Conformity assessment, risk management system, data gove..."
2,Limited Risk,"Chatbots, deepfakes, AI-generated content",Must disclose AI interaction to users. Deepfakes must be...
3,Minimal Risk,"Spam filters, AI in video games, recommendation systems",Encouraged to follow voluntary codes of conduct


## 5. Hallucination Mitigation Strategies

In [6]:
hallucination_strategies = pd.DataFrame([
    {
        "strategy": "RAG (Retrieval Augmented Generation)",
        "type": "Prevention",
        "how": "Ground every answer in retrieved real documents rather than model memory",
        "when_to_use": "Domain-specific Q&A where factual accuracy is critical"
    },
    {
        "strategy": "Temperature reduction",
        "type": "Prevention",
        "how": "Lower temperature (0.0-0.3) makes model more deterministic and less likely to confabulate",
        "when_to_use": "Factual extraction, classification, structured output tasks"
    },
    {
        "strategy": "Chain-of-thought prompting",
        "type": "Prevention",
        "how": "Force explicit reasoning steps before answering -- harder to hallucinate when reasoning is visible",
        "when_to_use": "Multi-step reasoning, math, logic tasks"
    },
    {
        "strategy": "Self-consistency sampling",
        "type": "Detection",
        "how": "Generate multiple answers at higher temperature, flag if they disagree significantly",
        "when_to_use": "High-stakes decisions where answer verification is worth the extra cost"
    },
    {
        "strategy": "LLM-as-judge (Day 54)",
        "type": "Detection",
        "how": "Use a second model call to evaluate faithfulness of the answer vs source context",
        "when_to_use": "RAG systems, any pipeline where output can be verified against a source"
    },
    {
        "strategy": "Uncertainty prompting",
        "type": "Prevention",
        "how": "Explicitly instruct model to say 'I don't know' rather than guessing",
        "when_to_use": "Any application where an honest 'I don't know' is better than a confident wrong answer"
    },
    {
        "strategy": "Output validation",
        "type": "Detection",
        "how": "Rule-based or model-based post-processing checks for known hallucination patterns",
        "when_to_use": "Structured outputs (dates, names, numbers) where validity can be checked programmatically"
    },
])

hallucination_strategies[["strategy", "type", "how"]]

,strategy,type,how
0,RAG (Retrieval Augmented Generation),Prevention,Ground every answer in retrieved real documents rather t...
1,Temperature reduction,Prevention,Lower temperature (0.0-0.3) makes model more determinist...
2,Chain-of-thought prompting,Prevention,Force explicit reasoning steps before answering -- harde...
3,Self-consistency sampling,Detection,"Generate multiple answers at higher temperature, flag if..."
4,LLM-as-judge (Day 54),Detection,Use a second model call to evaluate faithfulness of the ...
5,Uncertainty prompting,Prevention,Explicitly instruct model to say 'I don't know' rather t...
6,Output validation,Detection,Rule-based or model-based post-processing checks for kno...


## 6. Model Card — phi3-mini-ds-ai-finetuned

In [7]:
model_card = """
# Model Card: phi3-mini-ds-ai-finetuned

## Model Description
- **Base model:** microsoft/Phi-3-mini-4k-instruct (3.8B parameters)
- **Fine-tuning method:** QLoRA (4-bit quantization) + LoRA adapters via Unsloth
- **Developed by:** Vijendra Pokharkar (DS-AI-75D roadmap, Day 61)
- **Model type:** Causal Language Model (instruction-following)
- **Language:** English
- **HF Hub:** https://huggingface.co/VijendraHuggingface/phi3-mini-ds-ai-finetuned

## Intended Use
- **Primary use:** Answering Data Science and AI concept questions
- **Intended users:** Students and practitioners learning ML/AI fundamentals
- **Out-of-scope:** Medical advice, legal decisions, financial recommendations,
  production customer-facing applications without further evaluation

## Training Data
- 10 hand-crafted instruction-following examples covering:
  overfitting, precision/recall, transformers, RAG, gradient descent,
  dropout, vector databases, LoRA, attention mechanism,
  supervised vs unsupervised learning
- Format: Phi-3 instruction template (<|user|>/<|end|>/<|assistant|>)
- No personally identifiable information in training data

## Training Details
- Platform: Google Colab (Tesla T4 GPU, 15GB VRAM)
- LoRA rank (r): 16 | lora_alpha: 16
- Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
- Trainable parameters: 29,884,416 / 3,850,963,968 (0.78%)
- Epochs: 3 | Effective batch size: 8 | Learning rate: 2e-4
- Optimizer: AdamW 8-bit

## Evaluation
- Manual testing on held-out DS/AI questions
- Model correctly explained RAG, transformer architecture after fine-tuning
- No formal benchmark evaluation conducted (dataset too small for reliable metrics)

## Limitations
- Trained on only 10 examples — generalisation is very limited
- May hallucinate on topics not covered in training data
- Not evaluated on diverse demographic groups or languages
- Should not be used for high-stakes decisions without thorough evaluation

## Biases and Risks
- Training data was hand-crafted by a single author — may reflect individual
  perspective biases in explanations
- Base model (Phi-3-mini) may carry biases from its pretraining data
- Small fine-tuning dataset limits ability to detect or correct base model biases

## Responsible AI Considerations
- EU AI Act risk tier: Minimal Risk (educational/learning tool)
- No automated decision-making for consequential decisions
- Recommended: human review of any output used in educational materials

## How to Use
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    base = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3-mini-4k-instruct")
    model = PeftModel.from_pretrained(base,
                "VijendraHuggingface/phi3-mini-ds-ai-finetuned")
    tokenizer = AutoTokenizer.from_pretrained(
                "VijendraHuggingface/phi3-mini-ds-ai-finetuned")
"""

print(model_card)


# Model Card: phi3-mini-ds-ai-finetuned

## Model Description
- **Base model:** microsoft/Phi-3-mini-4k-instruct (3.8B parameters)
- **Fine-tuning method:** QLoRA (4-bit quantization) + LoRA adapters via Unsloth
- **Developed by:** Vijendra Pokharkar (DS-AI-75D roadmap, Day 61)
- **Model type:** Causal Language Model (instruction-following)
- **Language:** English
- **HF Hub:** https://huggingface.co/VijendraHuggingface/phi3-mini-ds-ai-finetuned

## Intended Use
- **Primary use:** Answering Data Science and AI concept questions
- **Intended users:** Students and practitioners learning ML/AI fundamentals
- **Out-of-scope:** Medical advice, legal decisions, financial recommendations,
  production customer-facing applications without further evaluation

## Training Data
- 10 hand-crafted instruction-following examples covering:
  overfitting, precision/recall, transformers, RAG, gradient descent,
  dropout, vector databases, LoRA, attention mechanism,
  supervised vs unsupervised learnin

## 7. Push Model Card to Hugging Face Hub

In [8]:
from huggingface_hub import HfApi
import os

api = HfApi(token=os.environ.get("HF_TOKEN") or os.environ.get("ANTHROPIC_API_KEY", ""))

# save model card locally
with open("day62_model_card.md", "w", encoding="utf-8") as f:
    f.write(model_card)

print("Model card saved locally as day62_model_card.md")
print("To push to HF Hub, run in terminal:")
print('  huggingface-cli upload VijendraHuggingface/phi3-mini-ds-ai-finetuned day62_model_card.md README.md')
print("\nOr paste your HF token directly:")
print("  api = HfApi(token='your-token-here')")
print("  api.upload_file(path_or_fileobj='day62_model_card.md', path_in_repo='README.md',")
print("                  repo_id='VijendraHuggingface/phi3-mini-ds-ai-finetuned')")

Model card saved locally as day62_model_card.md
To push to HF Hub, run in terminal:
  huggingface-cli upload VijendraHuggingface/phi3-mini-ds-ai-finetuned day62_model_card.md README.md

Or paste your HF token directly:
  api = HfApi(token='your-token-here')
  api.upload_file(path_or_fileobj='day62_model_card.md', path_in_repo='README.md',
                  repo_id='VijendraHuggingface/phi3-mini-ds-ai-finetuned')


In [ ]:
from huggingface_hub import HfApi

TOKEN = "your-hf-token-here"  # replace with your actual token when running

api = HfApi(token=TOKEN)

api.upload_file(
    path_or_fileobj="day62_model_card.md",
    path_in_repo="README.md",
    repo_id="VijendraHuggingface/phi3-mini-ds-ai-finetuned",
    token=TOKEN
)

print("Model card pushed successfully!")

c:\DS-AI-75d\.venv\Lib\site-packages\huggingface_hub\hf_api.py:11187: UserWarning: Warnings while validating metadata in README.md:
- empty or missing yaml metadata in repo card
  warnings.warn(f"Warnings while validating metadata in README.md:\n{message}")


Model card pushed successfully!


## 8. Summary — What We Covered Today

| Topic | Key Takeaway |
|-------|-------------|
| Bias types | 6 types: historical, representation, measurement, aggregation, evaluation, deployment |
| Fairness metrics | Demographic parity and equal opportunity gaps must be < 0.1 to pass |
| Our simulation | Both metrics FAILED (gaps of 0.295 and 0.273) — realistic example of a biased loan model |
| EU AI Act | 4 tiers: Unacceptable (banned), High Risk (regulated), Limited Risk (transparency), Minimal Risk (voluntary) |
| Our model's EU tier | Minimal Risk — educational tool, no consequential decisions |
| Hallucination mitigation | 7 strategies split between prevention (RAG, temperature, CoT) and detection (LLM-as-judge, self-consistency) |
| Model card | Written and pushed to HF Hub for phi3-mini-ds-ai-finetuned |

**Why this day matters for interviews:**
"How do you handle bias?" and "What is the EU AI Act?" are real questions
for DS/AI roles in the UK and Europe. Being able to name specific fairness
metrics (demographic parity, equal opportunity) and the EU AI Act risk tiers
puts you ahead of most candidates.